# Cropy 작업자 분리 시험셋: 원본 crop 효과 seed 반복 검증

실험 5에서 원본 해상도 crop으로 학습한 무 질병 전용 분류기(`hi448`)가 512 묶음 crop 학습(`low448`)보다 시험셋 정확도 54.5%→70.3%로 좋았습니다. 그런데 세 모델 모두 seed 1회였고 검증이 65장뿐이라 epoch 선택이 흔들립니다(768 입력이 448보다 나빴던 게 그 증거). 이 노트북은 **`low448`과 `hi448`을 seed 3개씩** 같은 설정으로 학습해 차이가 유지되는지 봅니다.

- 설정: ResNet50, 448 입력, 흐림 증강, 클래스 균형, lr 1e-4, 12 epoch, seed 1·2·3. 학습 스크립트는 `--seed`만 추가한 판(`train_full_colab_seed.py`).
- 판정: seed 3개 평균에서 `hi448`이 `low448`보다 정확도·균형 정확도 모두 높고, seed별 McNemar가 모두 같은 방향이면 "crop 품질 효과 확인"으로 봅니다.
- 결과: `MyDrive/cropy_testset/hirescrop_seeds_v1/` (`{low448,hi448}_s{1,2,3}.pth`, `test_report.json`)
- A100 기준 6회 학습 30분 안팎. 1·2번 데이터 준비는 이전 실험과 같아 같은 세션이면 바로 넘어갑니다.

**GPU 런타임(A100)** 에서 위에서부터 차례로 실행하세요.

In [ ]:
from pathlib import Path
import csv, hashlib, json, os, re, shutil, subprocess, sys, zipfile
import numpy as np, torch
from google.colab import drive

assert torch.cuda.is_available(), 'GPU 런타임을 선택한 뒤 다시 실행하세요.'
print('GPU:', torch.cuda.get_device_name(0))
drive.mount('/content/drive')
MY = Path('/content/drive/MyDrive')
TS = MY / 'cropy_testset'
RUN = TS / 'hirescrop_seeds_v1'
WORK = Path('/content/cropy_testset')
DATA = WORK / 'dataset'
SPLIT = WORK / 'split'
EPOCHS, BATCH_SIZE, LR = 12, 16, 1e-4
EXPECTED_BASE_SHA256 = '52dea0080285d76b6cbc09c2659c052a18f20ef620d05bca462203e64e6872ee'
EXPECTED_CODE_SHA256 = 'efe7034c6acd0544416e0cdb94c8fe748081921d73cf2fd1513fbc9b55e292a4'
SOURCES = {'base': MY / 'cropy_full_512/cropy_full_512.zip', 'code': MY / 'cropy_other10/training_code_other10.zip',
           'aihub': MY / 'cropy_other10/aihub_other_v1.zip', 'coco': MY / 'cropy_other10/coco_other_v1.zip',
           'server': MY / 'cropy_other10/cropy_other10_handoff_20261027/full512_resnet50_other10_v1.pth',
           'workers': TS / 'workers.json', 'trainer': TS / 'train_full_colab_seed.py',
           'baseline': TS / 'baseline_v1/best_model.pth', 'hires': TS / 'radish_hires_crops.zip'}
for path in SOURCES.values():
    assert path.is_file(), f'Drive에 파일이 없습니다: {path}'
WORK.mkdir(parents=True, exist_ok=True)
RUN.mkdir(parents=True, exist_ok=True)

## 1. 데이터 준비와 작업자 단위 재분할
Drive에서 로컬로 복사해 풀고, 모든 사진을 작업자 배정표대로 `split/{train,val,test}/<클래스>/`에 옮깁니다. AI Hub 9클래스 사진 수가 설계와 정확히 같은지 확인합니다.

In [ ]:
EXPECT = {"train": {"0": 589, "1": 528, "2": 267, "3": 124, "4": 574, "5": 310, "6": 5932, "7": 6726, "8": 5724}, "val": {"0": 134, "1": 166, "2": 59, "3": 49, "4": 107, "5": 58, "6": 1440, "7": 2023, "8": 1688}, "test": {"0": 361, "1": 330, "2": 212, "3": 78, "4": 230, "5": 125, "6": 2313, "7": 3349, "8": 3087}}

def digest(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def unpack(src, dest, expected=None):
    local = WORK / src.name
    if not local.exists():
        shutil.copyfile(src, local)
    if expected:
        assert digest(local) == expected, f'해시 불일치: {src.name}'
    with zipfile.ZipFile(local) as z:
        for item in z.infolist():
            assert (dest / item.filename).resolve().is_relative_to(dest.resolve()), '잘못된 압축 경로'
        z.extractall(dest)

unpack(SOURCES['code'], WORK / 'code', EXPECTED_CODE_SHA256)
ASSIGN = json.loads(SOURCES['workers'].read_text(encoding='utf-8'))['assign']
GROUP = re.compile(r'(?:^|_)([0-9A-Za-z]+)_(20\d{6})(?=_|\.|$)')
def split_of(path):
    m = GROUP.search(path.name)
    if m:
        return ASSIGN[m.group(1)] if m.group(1) in ASSIGN else 'train'
    if path.parent.name.startswith('9_'):  # COCO 조각: 내용 해시로 70/10/20
        b = int(digest(path)[:8], 16) % 10
        return 'train' if b < 7 else 'val' if b == 7 else 'test'
    return 'train'

if not (WORK / 'split_done').exists():
    shutil.rmtree(DATA, ignore_errors=True)
    shutil.rmtree(SPLIT, ignore_errors=True)
    unpack(SOURCES['base'], WORK, EXPECTED_BASE_SHA256)
    (DATA / 'manifest.csv').rename(DATA / 'manifest_9class.csv')
    unpack(SOURCES['aihub'], WORK)
    unpack(SOURCES['coco'], WORK)
    rows = []
    for src_split in ('train', 'val'):
        for path in sorted((DATA / 'images' / src_split).glob('*/*')):
            s = split_of(path)
            dest = SPLIT / s / path.parent.name / path.name
            dest.parent.mkdir(parents=True, exist_ok=True)
            shutil.move(str(path), dest)
            m = GROUP.search(path.name)
            rows.append([s, path.parent.name, path.name, m.group(1) if m else '', src_split])
    with (RUN / 'split_manifest.csv').open('w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['split', 'class', 'file', 'worker', 'official_split'])
        w.writerows(rows)
    (WORK / 'split_done').touch()

counts = {s: {c.name: len(list(c.iterdir())) for c in sorted((SPLIT / s).iterdir())} for s in ('train', 'val', 'test')}
for s, cs in counts.items():
    print(s, sum(cs.values()), cs)
for s, cs in EXPECT.items():
    for c, n in cs.items():
        got = counts[s][[k for k in counts[s] if k.startswith(c + '_')][0]]
        assert got == n, f'{s}/{c}: {got} != 설계 {n}'
print('AI Hub 9클래스 사진 수가 설계와 일치합니다.')
fingerprint = ''.join(f'{name}: {digest(WORK / SOURCES[name].name)}\n' for name in ('base', 'aihub', 'coco'))
fingerprint += f"workers: {digest(SOURCES['workers'])}\n"
(SPLIT.parent / 'manifest.csv').write_text(fingerprint, encoding='utf-8')  # 학습 스크립트는 data-dir 상위의 manifest.csv를 읽는다
from torchvision.datasets import ImageFolder
train_ds, val_ds = ImageFolder(SPLIT / 'train'), ImageFolder(SPLIT / 'val')
assert train_ds.classes == val_ds.classes and len(train_ds.classes) == 10, train_ds.classes
EXPECT_IMAGES = (len(train_ds), len(val_ds))
(RUN / 'split_counts.json').write_text(json.dumps(counts, ensure_ascii=False, indent=1), encoding='utf-8')
print('학습/검증:', EXPECT_IMAGES)

## 2. 두 가지 crop 데이터셋 준비
`LOW`는 512px 묶음에서 박스를 잘라 만들고(이전 실험과 같은 방식), `HI`는 업로드한 zip을 풉니다. 두 쪽에 모두 있는 사진만 남겨 비교를 공정하게 맞춥니다.

In [ ]:
from PIL import Image
MARGIN = 0.15
PAIR = ('2_무검은무늬병', '3_무노균병')
LOW, HI = WORK / 'crop_low', WORK / 'crop_hi'
label_index = {p.name[:-5]: p for p in (DATA / 'labels').rglob('*.json')}

def box_of(name):
    j = label_index.get(name)
    if j is None:
        return None
    d = json.loads(j.read_text(encoding='utf-8'))
    pts = [p for p in ((d.get('annotations') or {}).get('points') or []) if all(k in p for k in ('xtl', 'ytl', 'xbr', 'ybr'))]
    W, H = d['description'].get('width'), d['description'].get('height')
    if not pts or not W or not H:
        return None
    return (min(p['xtl'] for p in pts) / W, min(p['ytl'] for p in pts) / H, max(p['xbr'] for p in pts) / W, max(p['ybr'] for p in pts) / H)

# HI: zip 풀기 → crop_hi/images/{split}/{cls}/<stem>.jpg
if not (HI / 'done').exists():
    shutil.rmtree(HI, ignore_errors=True)
    with zipfile.ZipFile(SOURCES['hires']) as z:
        for item in z.infolist():
            assert (HI / item.filename).resolve().is_relative_to(HI.resolve()), '잘못된 압축 경로'
        z.extractall(HI)
    inner = HI / 'radish_hires_crops'
    (HI / 'images').mkdir(exist_ok=True)
    for s in ('train', 'val', 'test'):
        shutil.move(str(inner / s), HI / 'images' / s)
    (HI / 'done').touch()
hi_files = {s: {p.stem.lower(): p for c in PAIR for p in (HI / 'images' / s / c).glob('*')} for s in ('train', 'val', 'test')}

# LOW: 512 묶음에서 박스 crop → crop_low/images/{split}/{cls}/<원래 파일명>
if not (LOW / 'done').exists():
    shutil.rmtree(LOW, ignore_errors=True)
    dropped = 0
    for s in ('train', 'val', 'test'):
        for c in PAIR:
            out = LOW / 'images' / s / c
            out.mkdir(parents=True, exist_ok=True)
            for src in sorted((SPLIT / s / c).iterdir()):
                rel = box_of(src.name)
                if rel is None or src.stem.lower() not in hi_files[s]:
                    dropped += 1
                    continue
                with Image.open(src) as im:
                    im = im.convert('RGB'); w, h = im.size
                    bw, bh = (rel[2] - rel[0]) * w, (rel[3] - rel[1]) * h
                    l = max(0, int(rel[0] * w - MARGIN * bw)); t = max(0, int(rel[1] * h - MARGIN * bh))
                    r = min(w, int(rel[2] * w + MARGIN * bw)); b = min(h, int(rel[3] * h + MARGIN * bh))
                    im.crop((l, t, r, b)).save(out / src.name, quality=95)
    print('LOW에서 제외(박스 없음 또는 HI에 없음):', dropped)
    (LOW / 'done').touch()
low_files = {s: {p.stem.lower(): p for c in PAIR for p in (LOW / 'images' / s / c).glob('*')} for s in ('train', 'val', 'test')}
# HI 쪽도 LOW와 교집합만 남긴다
for s in ('train', 'val', 'test'):
    for stem, p in list(hi_files[s].items()):
        if stem not in low_files[s]:
            p.unlink(); del hi_files[s][stem]
    assert set(hi_files[s]) == set(low_files[s]), s
for d in (LOW, HI):
    shutil.copyfile(SPLIT.parent / 'manifest.csv', d / 'manifest.csv')
from torchvision.datasets import ImageFolder
counts = {}
for name, d in (('low', LOW), ('hi', HI)):
    tr, va = ImageFolder(d / 'images/train'), ImageFolder(d / 'images/val')
    assert tr.classes == va.classes == list(PAIR), tr.classes
    counts[name] = (len(tr), len(va), len((hi_files if name == 'hi' else low_files)['test']))
    print(name, 'train/val/test:', counts[name], {c: len(list((d / 'images/train' / c).iterdir())) for c in PAIR})
assert counts['low'] == counts['hi']
PAIR_IMAGES = counts['hi'][:2]
# 저장된 crop 크기 비교
import statistics
for name, d in (('low', LOW), ('hi', HI)):
    ls = [max(Image.open(p).size) for p in list(d.glob('images/test/*/*'))[:200]]
    print(name, 'test crop 긴 변 중앙값(px):', statistics.median(ls))

## 3. seed 3개 × 2 모델 학습
끊기면 다시 실행하면 완료된 모델은 건너뛰고 진행 중이던 모델은 마지막 epoch부터 이어집니다.

In [ ]:
shutil.copyfile(SOURCES['trainer'], WORK / 'code/train_full_colab.py')
SEEDS = (1, 2, 3)
MODELS = (('low448', LOW), ('hi448', HI))
for seed in SEEDS:
    for name, d in MODELS:
        tag = f'{name}_s{seed}'
        out = RUN / tag
        if (out / 'best_model.pth').exists() and (out / 'history.json').exists() and len(json.loads((out / 'history.json').read_text())) >= EPOCHS:
            print(tag, '이미 완료'); continue
        command = [sys.executable, '-u', str(WORK / 'code/train_full_colab.py'),
            '--data-dir', str(d / 'images'), '--output-dir', str(out),
            '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE), '--lr', str(LR), '--seed', str(seed),
            '--num-workers', '4', '--img-size', '448', '--arch', 'resnet50', '--blur-aug',
            '--expect-images', str(PAIR_IMAGES[0]), str(PAIR_IMAGES[1])]
        if (out / 'last_checkpoint.pth').exists():
            command.append('--resume')
        print('==', tag, '이어 학습:', '--resume' in command)
        subprocess.run(command, check=True)
        shutil.copyfile(out / 'best_model.pth', RUN / f'{tag}.pth')

## 4. seed별 평가와 요약
각 모델을 자기 자료의 test crop에 평가하고, seed별 McNemar와 seed 평균·표준편차를 냅니다.

In [ ]:
from torchvision import transforms
from math import comb
import statistics

def load(path):
    ckpt = torch.load(path, map_location='cpu', weights_only=True)
    sys.path.insert(0, str(WORK / 'code'))
    from train_full_colab import build_model
    model = build_model(ckpt.get('arch', 'resnet50'), ckpt['num_classes'], pretrained=False)
    model.load_state_dict(ckpt['model_state_dict'])
    return model.cuda().eval(), ckpt

def predict(model, size, files, bs=32):
    tf = transforms.Compose([transforms.Resize((size, size)), transforms.ToTensor(),
                             transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])
    out = []
    for i in range(0, len(files), bs):
        with torch.no_grad():
            out.append(model(torch.stack([tf(Image.open(p).convert('RGB')) for p in files[i:i + bs]]).cuda()).softmax(1).cpu().numpy())
    return np.concatenate(out)

def mcnemar(ca, cb):
    n01, n10 = int((ca & ~cb).sum()), int((~ca & cb).sum()); n = n01 + n10; k = min(n01, n10)
    return dict(only_first=n01, only_second=n10, p=round(min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n) if n else 1.0, 4))

stems = sorted(low_files['test'])
y = np.array([int(low_files['test'][s].parent.name[0]) for s in stems])
correct, report = {}, {'runs': {}}
for seed in SEEDS:
    for name, d in MODELS:
        tag = f'{name}_s{seed}'
        files = [(low_files if name == 'low448' else hi_files)['test'][s] for s in stems]
        model, ckpt = load(RUN / f'{tag}.pth')
        ids = [int(c.split('_', 1)[0]) for c in ckpt['classes']]
        probs = predict(model, ckpt.get('img_size', 448), files)
        pred = np.array(ids)[probs.argmax(1)]
        correct[tag] = pred == y
        r2 = float((pred[y == 2] == 2).mean()); r3 = float((pred[y == 3] == 3).mean())
        report['runs'][tag] = dict(epoch=int(ckpt['epoch']), accuracy=round(float((pred == y).mean()), 4), recall_2=round(r2, 4), recall_3=round(r3, 4), balanced=round((r2 + r3) / 2, 4))
        with (RUN / f'predictions_{tag}.csv').open('w', newline='', encoding='utf-8') as f:
            w = csv.writer(f); w.writerow(['file', 'label', 'pred'] + [f'p{i}' for i in ids])
            for p, yy, pp, pr in zip(files, y, pred, probs):
                w.writerow([p.name, int(yy), int(pp)] + [round(float(v), 5) for v in pr])
        print(tag, report['runs'][tag])
        del model
    report[f'mcnemar_s{seed}_low_vs_hi'] = mcnemar(correct[f'low448_s{seed}'], correct[f'hi448_s{seed}'])
summary = {}
for name, _ in MODELS:
    for key in ('accuracy', 'recall_2', 'recall_3', 'balanced'):
        vals = [report['runs'][f'{name}_s{s}'][key] for s in SEEDS]
        summary[f'{name}_{key}'] = dict(mean=round(statistics.mean(vals), 4), std=round(statistics.pstdev(vals), 4), values=vals)
report['summary'] = summary
report['pair_images'] = dict(train=PAIR_IMAGES[0], val=PAIR_IMAGES[1], test=int(len(y)))
(RUN / 'test_report.json').write_text(json.dumps(report, ensure_ascii=False, indent=1), encoding='utf-8')
for name, _ in MODELS:
    print(name, {k.split('_', 1)[1]: f"{v['mean']:.3f}±{v['std']:.3f}" for k, v in summary.items() if k.startswith(name)})
print({k: v for k, v in report.items() if k.startswith('mcnemar')})
print('저장:', RUN / 'test_report.json')